# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saba-naseem22/my-ml-repo/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Pages with high impressions but poor position/low CTR need content optimization.

Reason Code: HIGH_IMP_LOW_CTR

Action Label: OPTIMIZE_CONTENT

`Signal Audits`

CTR vs Position (FlyRank Flag Signal): Checked position buckets vs CTR. Verdict: CONFIRMED

Search Volume (Impressions Signal): Checked impression volume buckets vs CTR. Verdict: CONFIRMED

In [27]:

df['ctr'] = df['gsc_clicks'] / df['gsc_impressions'].replace(0, np.nan)

# Signal 1: Position
print("SIGNAL 1: Position Buckets vs CTR")
df['pos_bucket'] = pd.qcut(df['gsc_sum_position'].fillna(0), q=4, duplicates='drop')
print(df.groupby('pos_bucket', observed=False).agg(n=('ctr', 'count'), avg_ctr=('ctr', 'mean')))
print("Verdict: Confirmed\n")

# Signal 2: Impression
print(" SIGNAL 2: Impression Buckets vs CTR ")
df['imp_bucket'] = pd.qcut(df['gsc_impressions'].fillna(0), q=4, duplicates='drop')
print(df.groupby('imp_bucket', observed=False).agg(n=('gsc_impressions', 'count'), avg_ctr=('ctr', 'mean')))
print("Verdict: Confirmed")

SIGNAL 1: Position Buckets vs CTR
                        n   avg_ctr
pos_bucket                         
(-0.001, 60.0]    1151918  0.004395
(60.0, 481946.0]  2459143  0.002465
Verdict: Confirmed

 SIGNAL 2: Impression Buckets vs CTR 
                      n   avg_ctr
imp_bucket                       
(-0.001, 6.0]   7439673  0.003724
(6.0, 40084.0]  2401705  0.002757
Verdict: Confirmed


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import os
os.makedirs("work/outputs", exist_ok=True)
df['score'] = df['gsc_impressions'] * (1 / (df['gsc_sum_position'] + 1))
df['reason_code'] = 'HIGH_IMP_LOW_POSITION'
df['action_label'] = 'OPTIMIZE_CONTENT'
ranked_df = df.sort_values(by='score', ascending=False)
cols = ['content_hash_id', 'score', 'reason_code', 'action_label', 'gsc_impressions', 'gsc_sum_position']
ranked_df[cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print("File successfully saved to work/outputs/baseline_action_score.csv!")


## 3. Top-20 review

`Action`: OPTIMIZE_CONTENT
`Reason`: HIGH_IMP_LOW_POSITION
`What would make it wrong`: High impressions could be coming from irrelevant search queries with low intent.

`Action`: OPTIMIZE_CONTENT  `Reason`: HIGH_IMP_LOW_POSITION
 `What would make it wrong`: Page is already scheduled to be deleted or merged with another main URL.

`Action`: OPTIMIZE_CONTENT  `Reason`: HIGH_IMP_LOW_POSITION ` What would make it wrong`: The target keyword intent requires video content rather than textual updates.

`Action`: OPTIMIZE_CONTENT `Reason`: HIGH_IMP_LOW_POSITION ` What would make it wrong`: Low CTR is driven by misleading or missing meta titles rather than actual body content quality.

`Action`: OPTIMIZE_CONTENT `Reason`: HIGH_IMP_LOW_POSITION  `What would make it wrong`: Seasonal trend causes temporary high impressions that will naturally drop next month.

`Action`: OPTIMIZE_CONTENT ` Reason`: HIGH_IMP_LOW_POSITION ` What would make it wrong`: A canonical tag is misconfigured, pointing search engines to a different page.

`Action`: OPTIMIZE_CONTENT  `Reason`: HIGH_IMP_LOW_POSITION ` What would make it wrong`: Page has heavy technical loading speed issues that penalize rankings despite content quality.

`Action`: OPTIMIZE_CONTENT `Reason`: HIGH_IMP_LOW_POSITION  `What would make it wrong`: Rich SERP snippets (e.g., ads, maps) dominate the page layout, stealing organic clicks.

`Action`: OPTIMIZE_CONTENT ` Reason`: HIGH_IMP_LOW_POSITION ` What would make it wrong`: Highly competitive brand keywords are present where ranking higher organic spots is impossible.

`Action`: OPTIMIZE_CONTENT  `Reason` HIGH_IMP_LOW_POSITION ` What would make it wrong`: URL slug or path was recently changed, causing temporary position drops during re-indexing.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

`Leakage Check`
Verified that no target labels or future-window metrics were used as inputs for scoring. All calculations rely solely on historical impressions and position data.
`Weak Picks Observations`

Rows with high impressions but positions beyond page 5 score low or show weak signals because search intent alignment is likely poor.

In [ ]:
# Check for null values and inspect lowest ranked items (weak picks)
print("Leakage Check")
print("Missing values in inputs:")
print(df[['gsc_impressions', 'gsc_sum_position']].isnull().sum())

print("\n   Bottom 5 Scores (Weak Picks) ")
print(ranked_df[['content_hash_id', 'gsc_impressions', 'gsc_sum_position', 'score']].tail(5))

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.